In [1]:
import requests
import pandas as pd
import numpy as np
import random
import hashlib
import json
from langchain_openai import ChatOpenAI
from langchain_community.embeddings import XinferenceEmbeddings
from ragas.llms import LangchainLLMWrapper
from ragas.embeddings import LangchainEmbeddingsWrapper
from ragas import EvaluationDataset, SingleTurnSample
from ragas import evaluate, metrics
from ragas.run_config import RunConfig
from pydantic import SecretStr
from tqdm import tqdm
from loguru import logger
from pathlib import Path
from datetime import datetime

In [2]:
# 读取 TEST 数据集
logger.info('Loading TEST JSONL file...')
df = pd.read_json("qac_dataset_test.jsonl", lines=True, encoding="utf-8")
logger.success("TEST JSONL file is read.")

# 创建 List 存储 QAC 数据
logger.info('Creating QAC list...')
qac_list = []
for index, row in df.iterrows():
    qac_list.append({
        "question": row["question"],
        "answer": row["answer"],
        "context": row["context"]
    })
logger.success(f"QAC list is created. Total QAC size: {len(qac_list)}")

2025-06-22 23:57:26.745 | INFO     | __main__:<module>:2 - Loading TEST JSONL file...
2025-06-22 23:57:26.755 | SUCCESS  | __main__:<module>:4 - TEST JSONL file is read.
2025-06-22 23:57:26.755 | INFO     | __main__:<module>:7 - Creating QAC list...
2025-06-22 23:57:26.763 | SUCCESS  | __main__:<module>:15 - QAC list is created. Total QAC size: 200


In [ ]:
base_url = "http://10.26.58.108/v1/chat-messages"
api_key = "app-xzQMDIdrgdDTwxDxGHe31iBd"

headers = {
    'content-type': 'application/json; charset=UTF-8',
    'Authorization': f'Bearer {api_key}',
}
logger.success("Dify API request header is built.")

# 除去前 113 个元素
qac_list = qac_list[113:]

# 提交 QAC 数据集
logger.info('Submitting QAC items to Dify...')
samples = []
for qac in tqdm(
    qac_list, desc="Submitting QAC items to Dify",
    unit="item", total=len(qac_list)
):
    # 拿到 QAC 数据
    question = qac.get("question", "")
    reference = qac.get("answer", "")
    reference_contexts = [qac.get("context", "")]

    # 构建 QAC API 请求参数
    payload = {
        "inputs": {},
        "query": question,
        "response_mode": "blocking",
        "conversation_id": "",
        "user": "abc-123",
        "files": []
    }

    # 提取 metadata 里面 retriever_resources 中的 content 作为召回文本块
    response = requests.post(
        url=base_url,
        headers=headers,
        json=payload,
    )

    res = response.json()

    # 每经过一次循环就保存成本地文件，在 logs 文件夹中
    try:
        LOG_DIR = Path("logs")
        LOG_DIR.mkdir(exist_ok=True)

        timestamp = datetime.now().strftime("%Y%m%d_%H%M%S_%f")
        content_hash = hashlib.md5(json.dumps(res).encode()).hexdigest()[:6]
        filename = f"{timestamp}_{content_hash}.json"
        log_path = LOG_DIR / filename
        with open(log_path, 'w', encoding='utf-8') as f:
            json.dump({
                "metadata": {
                    "save_time": datetime.now().isoformat(),
                    "source": "api_response"
                },
                "data": res
            }, f, indent=2, ensure_ascii=False)
    except Exception as e:
        print(f"❌ 保存失败: {e}")
        raise

    retriever_resources = res.get('metadata', {}).get('retriever_resources', [])
    retriever_context = [
        doc.get("content", "") for doc in retriever_resources
    ]
    response = res.get('answer', "")

    # 创建 SingleTurnSample 对象
    # user_input: 用户输入的问题
    # retrieved_contexts: AI 召回的相关文本
    # response: AI 生成的回答
    # reference_contexts: 人给出的正确召回片段
    # reference: 人给出的正确回答
    sample = SingleTurnSample(
        user_input=question,
        retrieved_contexts=retriever_context,
        response=response,
        reference_contexts=reference_contexts,
        reference=reference
    )

    # 将样本添加到列表
    samples.append(sample)

logger.success('Submitted QAC items to Dify.')

2025-06-22 23:57:28.798 | SUCCESS  | __main__:<module>:8 - Dify API request header is built.
2025-06-22 23:57:28.799 | INFO     | __main__:<module>:14 - Submitting QAC items to Dify...
Submitting QAC items to Dify:   8%|▊         | 7/87 [13:00<2:28:36, 111.45s/item]

In [ ]:
try:
    logger.info("Creating the LLM and Embeddings.")
    # 创建 LLM 和 Embeddings 模型
    evaluator_llm = LangchainLLMWrapper(
        ChatOpenAI(
            model="deepseek-chat",
            api_key=SecretStr("sk-574d077e01be45beab39804304a15109"),
            base_url="https://api.deepseek.com/v1",
            temperature=0,
            max_tokens=None,
            timeout=None,
            max_retries=2,
        )
    )
    logger.success(f"LLM successfully created.")
    # 用于评估的嵌入模型
    evaluator_embeddings = LangchainEmbeddingsWrapper(
        XinferenceEmbeddings(
            server_url="http://10.26.58.108:9998",
            model_uid="bge-m3-MsvUdbGI"
        )
    )
    logger.success(f"Embeddings successfully created.")
except Exception as e:
    logger.exception(f"Failed to initialize LLM or Embeddings: {e}")
    raise

In [ ]:
# 将最后的结果添加到 EvaluationDataset 中
eval_dataset = EvaluationDataset(samples=samples)

# 评估指标列表
metrics_list = [
    # 答案正确性，通过嵌入模型比较答案的相似程度
    metrics.answer_correctness,
    # 答案相关性，越是不完整或包含冗余信息的答案，得分越低
    metrics.answer_relevancy,
    # 忠实度/可信度，衡量了生成的答案与给定上下文的事实一致性
    metrics.faithfulness,
    # 上下文精度，评估所有在上下文中呈现的与基本事实相关的条目是否排名较高。
    metrics.context_precision,
    # 上下文召回率，衡量检索到的上下文与人类提供的真实答案的一致程度。
    metrics.context_recall,
]

# 创建 run config
eval_run_config = RunConfig(timeout=600, log_tenacity=True)

# 评估数据集
logger.info('Evaluating QAC items...')
results = evaluate(
    dataset=eval_dataset,
    metrics=metrics_list,
    llm=evaluator_llm,
    embeddings=evaluator_embeddings,
    run_config=eval_run_config
)
logger.success('Evaluated QAC items.')

# 保存结果到 CSV 文件
logger.info('Saving results to CSV file...')
results.to_pandas().to_csv("qac_results.csv")
logger.success('Results saved to CSV file.')